In [4]:
import os
import glob
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoVideoProcessor, AutoModel
from peft import LoraConfig, get_peft_model, TaskType
from PIL import Image
from tqdm import tqdm

# Setup device and data types for optimal performance on MPS
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
dtype = torch.float32 # MPS works best with float32 precision

print(f"✅ Using device: {device} | Ready for LoRA Fine-tuning")

✅ Using device: mps | Ready for LoRA Fine-tuning


In [5]:
class RugbyVideoDataset(Dataset):
    def __init__(self, root_dir, processor, num_frames=16):
        self.root_dir = root_dir
        self.processor = processor
        self.num_frames = num_frames
        
        self.labels = ["Tackle", "Ruck", "Maul", "General Play", "Out of Play", "Scrum", "Lineout", "Goal Kick"]
        self.label2id = {label: i for i, label in enumerate(self.labels)}
        
        self.video_paths = []
        self.video_labels = []
        
        for label in self.labels:
            folder_path = os.path.join(root_dir, label)
            mp4_files = glob.glob(os.path.join(folder_path, "*.mp4"))
            for f in mp4_files:
                self.video_paths.append(f)
                self.video_labels.append(self.label2id[label])
                
        print(f"📦 Total videos: {len(self.video_paths)} ready for Fine-tuning.")

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        cap = cv2.VideoCapture(self.video_paths[idx])
        pil_frames = []
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        step = max(1, total_frames // self.num_frames)
        
        for i in range(self.num_frames):
            cap.set(cv2.CAP_PROP_POS_FRAMES, min(i * step, total_frames - 1))
            ret, frame = cap.read()
            if not ret: break
            pil_frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        cap.release()

        while len(pil_frames) < self.num_frames:
            pil_frames.append(pil_frames[-1] if pil_frames else Image.new('RGB', (224, 224)))

        inputs = self.processor(pil_frames, return_tensors="pt")
        return inputs['pixel_values_videos'].squeeze(0), torch.tensor(self.video_labels[idx])

In [6]:
# 1. Load backbone model
base_model = AutoModel.from_pretrained("facebook/vjepa2-vitl-fpc64-256")

# 2. Configure LoRA
lora_config = LoraConfig(
    r=8,                # Rank
    lora_alpha=32,       # Scaling factor
    target_modules=["query", "value"], # Modules to apply LoRA
    lora_dropout=0.1,
    bias="none",
)

# 3. Insert LoRA into backbone
lora_vjepa = get_peft_model(base_model, lora_config)

# 4. Define final Model
class RugbyLoraClassifier(nn.Module):
    def __init__(self, lora_backbone, num_classes=8):
        super().__init__()
        self.backbone = lora_backbone
        embed_dim = self.backbone.config.hidden_size
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.5),
            nn.Linear(embed_dim, num_classes)
        )
        
        # Load pre-trained linear head weights if available
        if os.path.exists("rugby_linear_head.pth"):
            print("🚀 Loading pre-trained linear head weights...")
            self.head.load_state_dict(torch.load("rugby_linear_head.pth", map_location=device))

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values_videos=pixel_values)
        # Global average pooling over the temporal dimension
        features = outputs.last_hidden_state.mean(dim=1)
        return self.head(features)

model = RugbyLoraClassifier(lora_vjepa).to(device)
model.backbone.print_trainable_parameters()

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

🚀 Loading pre-trained linear head weights...
trainable params: 933,888 || all params: 326,905,216 || trainable%: 0.2857


In [7]:
DATA_DIR = "mydata"
BATCH_SIZE = 4
NUM_WORKERS = 0

processor = AutoVideoProcessor.from_pretrained("facebook/vjepa2-vitl-fpc64-256")
full_ds = RugbyVideoDataset(DATA_DIR, processor, num_frames=16)

train_size = int(0.8 * len(full_ds))
val_size = len(full_ds) - train_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"🔥 Ready to train with {len(train_ds)} training samples!")

📦 Total videos: 366 ready for Fine-tuning.
🔥 Ready to train with 292 training samples!


In [8]:
EPOCHS = 10
LEARNING_RATE = 5e-5 # 20 times smaller than Linear Probing to allow LoRA to fine-tune effectively without overshooting

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

print("🎯 Bắt đầu LoRA Fine-tuning...")

for epoch in range(EPOCHS):
    model.train()
    correct, total, epoch_loss = 0, 0, 0
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for videos, labels in loop:
        videos, labels = videos.to(device), labels.to(device)
        
        optimizer.zero_grad()
        logits = model(videos)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        _, pred = logits.max(1)
        correct += pred.eq(labels).sum().item()
        total += labels.size(0)
        loop.set_postfix(acc=100.*correct/total, loss=loss.item())

    # Validation after each epoch
    model.eval()
    v_correct, v_total = 0, 0
    with torch.no_grad():
        for v_videos, v_labels in val_loader:
            v_videos, v_labels = v_videos.to(device), v_labels.to(device)
            v_logits = model(v_videos)
            _, v_pred = v_logits.max(1)
            v_correct += v_pred.eq(v_labels).sum().item()
            v_total += v_labels.size(0)
    
    print(f"🌟 Epoch {epoch+1} Val Acc: {100.*v_correct/v_total:.2f}%")

# Save the final LoRA fine-tuned model
torch.save(model.state_dict(), "rugby_vjepa_lora_final.pth")
print("✅ Fine-tuning complete! Final model saved as 'rugby_vjepa_lora_final.pth'")

🎯 Bắt đầu LoRA Fine-tuning...


Epoch 1: 100%|██████████| 73/73 [17:54<00:00, 14.72s/it, acc=62.7, loss=2.64] 


🌟 Epoch 1 Val Acc: 67.57%


Epoch 2: 100%|██████████| 73/73 [07:48<00:00,  6.41s/it, acc=63, loss=0.673]  


🌟 Epoch 2 Val Acc: 67.57%


...

🌟 Epoch 8 Val Acc: 89.19%


Epoch 9: 100%|██████████| 73/73 [07:03<00:00,  5.80s/it, acc=83.9, loss=0.484] 


🌟 Epoch 9 Val Acc: 91.89%


Epoch 10: 100%|██████████| 73/73 [07:00<00:00,  5.77s/it, acc=86.6, loss=0.0749]


🌟 Epoch 10 Val Acc: 94.59%
✅ Fine-tuning complete! Final model saved as 'rugby_vjepa_lora_final.pth'
